In [1]:
import pandas as pd
import numpy as np

customers = pd.read_csv("./data/raw/customers.csv")

customers.info()

<class 'pandas.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   customer_id  150 non-null    int64
 1   name         150 non-null    str  
 2   gender       150 non-null    str  
 3   age          150 non-null    int64
 4   city         150 non-null    str  
 5   signup_date  150 non-null    str  
dtypes: int64(2), str(4)
memory usage: 11.0 KB


다변량 EDA

In [2]:
order_items = pd.read_csv("./data/raw/order_items.csv")

order_items.info()

<class 'pandas.DataFrame'>
RangeIndex: 764 entries, 0 to 763
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   order_item_id  764 non-null    int64
 1   order_id       764 non-null    int64
 2   product_id     764 non-null    int64
 3   quantity       764 non-null    int64
 4   unit_price     764 non-null    int64
dtypes: int64(5)
memory usage: 30.0 KB


In [3]:
orders = pd.read_csv("./data/raw/orders.csv")

orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   order_id        300 non-null    int64
 1   customer_id     300 non-null    int64
 2   order_date      300 non-null    str  
 3   payment_method  300 non-null    str  
 4   order_status    300 non-null    str  
dtypes: int64(2), str(3)
memory usage: 19.9 KB


In [4]:
products = pd.read_csv("./data/raw/products.csv")

products.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   product_id    100 non-null    int64
 1   product_name  100 non-null    str  
 2   category      100 non-null    str  
 3   price         100 non-null    int64
dtypes: int64(2), str(2)
memory usage: 6.0 KB


In [7]:
#날짜 타입

orders["order_date"] = pd.to_datetime(orders["order_date"], errors="coerce")
orders["order_date"].info()

<class 'pandas.Series'>
RangeIndex: 300 entries, 0 to 299
Series name: order_date
Non-Null Count  Dtype         
--------------  -----         
300 non-null    datetime64[us]
dtypes: datetime64[us](1)
memory usage: 2.5 KB


In [11]:
print(customers["customer_id"].isna().sum())
print(order_items["order_item_id"].isna().sum())
print(orders["order_id"].isna().sum())
print(products["product_id"].isna().sum())

0
0
0
0


In [16]:
#키 중복 확인

for frame, key in [(customers,"customer_id"), (order_items,"order_item_id"), (orders,"order_id"), (products,"product_id")]:
    print(key, "결측 개수:", frame[key].isna().sum())

customer_id 결측 개수: 0
order_item_id 결측 개수: 0
order_id 결측 개수: 0
product_id 결측 개수: 0


In [45]:
#파생컬럼 total_prce(quantity * unit_price) 를 order_items에 추가
order_items["total_price"] = order_items["quantity"] * order_items["unit_price"]

In [21]:
#도시, 섭엽ㄹ, 나이로 등록 고객의 분포(value_count,describe)를 확인해 보세요

print(customers["city"].value_counts())
print(customers["gender"].value_counts())
print(customers["age"].describe())

city
성남    21
광주    17
부산    16
대구    15
서울    15
울산    14
인천    14
대전    14
수원    13
고양    11
Name: count, dtype: int64
gender
F    84
M    66
Name: count, dtype: int64
count    150.000000
mean      42.086667
std       15.613166
min       19.000000
25%       29.000000
50%       40.000000
75%       57.000000
max       69.000000
Name: age, dtype: float64


In [28]:
# 상품 카테고리의 개수와 가격 통계
product_count = products["category"].value_counts(dropna=False).to_frame()
print(type(product_count))
product_count

<class 'pandas.DataFrame'>


,count
category,
스포츠,19
전자기기,17
생활용품,16
뷰티,16
도서,14
패션,11
식품,7


In [29]:
products["price"].describe()

count       100.000000
mean     110040.000000
std       56433.910574
min        5000.000000
25%       65750.000000
50%      112000.000000
75%      161000.000000
max      200000.000000
Name: price, dtype: float64

In [32]:
# 상품 카테고리별 분표

category_price = products.groupby("category", dropna=False).agg(
    product_count = ("product_id", "size"),
    mean_price = ("price", "mean"),
    median_price = ("price", "median")
)

print(category_price)

          product_count     mean_price  median_price
category                                            
도서                   14  106857.142857      118500.0
뷰티                   16  117687.500000      134000.0
생활용품                 16   96437.500000       89000.0
스포츠                  19  111578.947368      103000.0
식품                    7  137142.857143      147000.0
전자기기                 17  101588.235294      111000.0
패션                   11  115909.090909      115000.0


완료 주문 병합 후 검증

In [35]:
#불리언마스크
#orders["order_status"] == "completed"
#completed_orders = orders.value_counts("order_status",)
completed_orders = orders[orders["order_status"] == "completed"]
print(completed_orders["order_status"].value_counts())

order_status
completed    184
Name: count, dtype: int64


In [46]:
# order_items에서 completed 인 항목
order_items.merge(
    orders[["order_id", "customer_id", "order_date", "order_status"]],
    on="order_id", how="left", validate="many_to_one"
).head()


,order_item_id,order_id,product_id,quantity,unit_price,total_price,customer_id,order_date,order_status
0,1,1,100,3,102000,306000,123,2026-05-07,completed
1,2,1,87,5,25000,125000,123,2026-05-07,completed
2,3,1,7,3,142000,426000,123,2026-05-07,completed
3,4,1,9,3,193000,579000,123,2026-05-07,completed
4,5,2,72,4,189000,756000,77,2025-07-23,cancelled


In [48]:
completed_item_with_orders = item_with_orders[
    item_with_orders["order_status"] == "completed"
]

NameError: name 'item_with_orders' is not defined

In [ ]:
#고객별 평균 주문건수(완료건기준)

# 고객별 완료 주문 건수
customer_completed_orders = (
    completed_item_with_orders
    .groupby("customer_id")["order_id"]
    .nunique()
)

customer_completed_orders.head()

NameError: name 'item_with_orders' is not defined

In [47]:
#고객별 완료 주문 건수?
customer_completed_orders = (
    completed_item_with_orders
    .groupby("customer_id")["order_id"]
    .nunique()
)

customer_completed_orders.head()


NameError: name 'completed_item_with_orders' is not defined

In [ ]:
# 카테고리별 수량 합산하기, 완료주문 금액 합산해보세요